# Traceback translator

Translate the sentence in a Python traceback into an Indian language, leave every
technical token exactly where it was, and prove it afterwards.

**This notebook has not been run against the live API.** There was no
`SARVAM_API_KEY` on the machine where it was written, so every code cell below
ships with an empty output. Nothing was executed and no result was invented. Run
it yourself with a key to see real output.

A student runs their program and gets this:

```
Traceback (most recent call last):
  File "assignment2.py", line 14, in average
    return total / count
           ~~~~~~^~~~~~~
ZeroDivisionError: division by zero
```

Almost all of that must never change: the file name, the line number, the
function name, the echoed line of their own code, the caret that points at the
failing operator, and the exception class. One phrase is not technical --
**division by zero** -- and it is the only part that explains what went wrong.

The pipeline is five steps:

```
capture  ->  parse  ->  mask  ->  translate  ->  restore  ->  check
```

If the check finds that a single frame line, code echo, caret line, chain note
or exception class moved, the recipe shows the original traceback unchanged and
says why. A partly translated traceback is never shown to anyone.

In [ ]:
%pip install -q -r requirements.txt

## 1. Setup

The parser, the masker, the restorer and the integrity check need no account at
all. Only the translate step needs a key, and it is not used until section 5.

In [ ]:
from __future__ import annotations

import sys
import traceback
from pathlib import Path

RECIPE_DIR = Path.cwd()
if str(RECIPE_DIR) not in sys.path:
    sys.path.insert(0, str(RECIPE_DIR))

from traceback_translator import (
    SENTINEL_RE,
    UnsupportedTracebackError,
    mask_message,
    message_skip_reason,
    parse_traceback,
    render_traceback,
    restore_message,
    translate_traceback,
    verify_integrity,
)

TARGET_LANGUAGE = "hi-IN"

print("offline core loaded; target language:", TARGET_LANGUAGE)

## 2. The corpus generates itself

There is no input file, no download and no licence question. Every fixture below
is produced by running a small broken snippet inside `try` / `except` and
capturing `traceback.format_exc()`.

That means the tracebacks you see are the ones **your** interpreter produced. On
a different Python version they will look different -- older interpreters print
no caret and tilde anchor lines at all -- and that is correct, not a bug.

In [ ]:
def capture(broken) -> str:
    """Run a broken snippet and return the traceback this interpreter printed."""
    try:
        broken()
    except BaseException:
        return traceback.format_exc()
    raise AssertionError("that snippet did not raise, so there is nothing to show")


def divide_by_zero() -> None:
    total, count = 10, 0
    return total / count


def missing_key() -> None:
    marks = {"name": "Asha"}
    return marks["user_id"]


def bad_number() -> None:
    return int("abc")


def chained() -> None:
    try:
        1 / 0
    except ZeroDivisionError as exc:
        raise RuntimeError("could not compute the average") from exc


CORPUS = {
    "divide_by_zero": capture(divide_by_zero),
    "missing_key": capture(missing_key),
    "bad_number": capture(bad_number),
    "chained": capture(chained),
}

for name, text in CORPUS.items():
    print("=" * 70)
    print(name)
    print("=" * 70)
    print(text)

## 3. Parse

`parse_traceback` splits the text into segments -- one per exception in a chain
-- and pulls out the frames, the exception class and the human message.

Two facts drive the parser and both are the opposite of the obvious guess.

**The last line of a traceback is not the exception line.** A message containing
a newline renders across two physical lines, so `text.splitlines()[-1]` returns
a line with no exception class in it at all. The parser walks forward from the
frames instead.

**`raise ValueError("")` and `raise ValueError` render identically.** Both print
the bare line `ValueError`. There is no way to tell an empty message from an
absent one in the rendered text, so this recipe does not try: both are "no
message" and neither is sent anywhere.

In [ ]:
for name, text in CORPUS.items():
    parsed = parse_traceback(text)
    print("=" * 70)
    print(name, "--", len(parsed.segments), "segment(s)")
    for note in parsed.chain_notes:
        print("   joined by:", note)
    for segment in parsed.segments:
        print("   class  :", segment.exception_class)
        print("   message:", repr(segment.message))
        print("   frames :", len(segment.frames))
        for frame in segment.frames:
            print("      ", Path(frame.path).name, "line", frame.lineno, "in", frame.func)

### The identity render is byte-exact

`render_traceback` rebuilds the whole traceback from the parse. Handing it a
replacement of `None` for every segment means "change nothing", and the result
has to equal the input byte for byte, trailing newline included. If that ever
stops being true, nothing downstream can be trusted.

In [ ]:
for name, text in CORPUS.items():
    parsed = parse_traceback(text)
    rebuilt = render_traceback(parsed, [None] * len(parsed.segments))
    print(f"{name:20s} round trip exact: {rebuilt == text}")

## 4. Mask

Everything technical inside the message is replaced with a numbered sentinel
before anything leaves the machine. A span is protected when it is quoted, or
bracketed, or looks like a path, or is a call form such as `len()`, or a dunder,
or a dotted name, or one of five Python literals, or a type name that is not
also an English word, or a word carrying an uppercase letter after the first
position, or a word carrying a digit or an underscore.

Note what is **not** protected. `list`, `set`, `type`, `object` and `range` are
all builtin types and all ordinary English words, and CPython uses them as
ordinary English in its own messages -- `IndexError: list index out of range`
contains two of them. Protecting everything in `builtins` would freeze that
message solid and translate nothing useful.

Some messages need no call at all. `KeyError: 'user_id'` masks down to a single
sentinel, so there is nothing left to translate and no reason to spend anything
on it.

In [ ]:
for name, text in CORPUS.items():
    parsed = parse_traceback(text)
    for segment in parsed.segments:
        if segment.message is None:
            continue
        masked = mask_message(segment.message)
        print("original :", segment.message)
        print("masked   :", masked.masked)
        print("tokens   :", masked.tokens)
        print("skip     :", message_skip_reason(segment.message))
        print("reverses :", restore_message(masked.masked, masked.tokens) == segment.message)
        print("-" * 70)

## 5. Translate

This is the only step that needs a key and the only step that leaves the
machine. The parameter set is fixed and each value has a reason, all read from
the SDK's own docstring for `text.translate`:

- `model="sarvam-translate:v1"` -- it covers all 22 scheduled languages of India
  and allows 2000 input characters.
- `mode="formal"` -- the only mode this model supports.
- `numerals_format="international"` -- native numerals would rewrite the digit
  inside a sentinel. It is already the default, but the sentinel scheme depends
  on it and a default is not a guarantee, so it is passed explicitly.
- `output_script` is never passed -- transliteration is not supported for this
  model.

`mayura:v1` is the alternative: 12 languages, a 1000-character cap, and more
modes. This recipe stays on the wider model because a reader who needs Maithili,
Santali or Bodo is exactly who it is for.

The key is passed explicitly. Relying on the client's own default argument is
the mistake this repository has fixed before.

`sarvam_translation.py` in this directory holds the same call packaged as an
importable function, if you would rather use it from your own tool than from a
notebook.

In [ ]:
import os

from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

if not os.environ.get("SARVAM_API_KEY"):
    raise RuntimeError(
        "SARVAM_API_KEY is not set. Copy .env.example to .env and put your key in it."
    )

client = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])
print("client ready")

In [ ]:
def translate(masked: str) -> str:
    """One masked message in, one translated message out."""
    if not any(char.isalpha() for char in SENTINEL_RE.sub("", masked)):
        return masked          # nothing but sentinels and punctuation: no call, no cost

    response = client.text.translate(
        input=masked,
        source_language_code="en-IN",
        target_language_code=TARGET_LANGUAGE,
        model="sarvam-translate:v1",
        mode="formal",
        numerals_format="international",
    )
    return response.translated_text

## 6. Restore, then check

`translate_traceback` runs the whole pipeline: mask, translate, restore, rebuild,
and then hand the result to the integrity check.

The check compares the rebuilt traceback against the original line by line. Every
line that is not an exception-line message must be byte-identical, and every span
that was protected in an original message must still be present. If anything
fails, the original text comes back unchanged along with a named reason.

In [ ]:
for name, text in CORPUS.items():
    result = translate_traceback(text, translate)
    print("=" * 70)
    print(name)
    print("=" * 70)
    print("messages translated:", result.translated_count)
    print("messages skipped   :", result.skipped)
    print("integrity failures :", [failure.reason for failure in result.failures])
    print()
    print("--- before ---")
    print(text)
    print("--- after ----")
    print(result.text)

### What the check catches

The check is the product; everything else is plumbing around it. Below, a
deliberately bad translator rewrites the whole message and drops the quoted
identifier that was supposed to be protected. The result is refused, and the
original traceback comes back untouched.

In [ ]:
def careless(masked: str) -> str:
    """A translator that ignores the sentinels and rewrites everything."""
    return "sab kuch badal diya"


text = CORPUS["missing_key"]
result = translate_traceback(text, careless)

print("returned the original unchanged:", result.text == text)
for failure in result.failures:
    print(failure.reason, "--", failure.detail)

In [ ]:
sabotaged = CORPUS["divide_by_zero"].replace("line ", "line 9999 ", 1)
for failure in verify_integrity(CORPUS["divide_by_zero"], sabotaged):
    print(failure.reason, "--", failure.detail)

### Shapes this version refuses by name

An exception group prints a different document -- gutter characters, numbered
rules and an indented sub-block per child -- so it is refused rather than
half-parsed. A message that spans more than one physical line is refused too: the
line count is part of the traceback's structure, and a translator that joins or
splits lines breaks it silently.

In [ ]:
def two_things_wrong() -> None:
    raise ExceptionGroup("two things went wrong", [ValueError("bad value"), KeyError("k")])


def message_on_two_lines() -> None:
    raise ValueError("first line\nsecond line")


try:
    parse_traceback(capture(two_things_wrong))
except UnsupportedTracebackError as exc:
    print("refused:", exc.reason, "--", exc.detail)

parsed = parse_traceback(capture(message_on_two_lines))
segment = parsed.segments[0]
print("message lines:", segment.message_line_count)
print("refused:", message_skip_reason(segment.message))

## 7. Save the result

The translated traceback is written to `outputs/`, which is gitignored, so
nothing you run here can end up in a commit by accident.

In [ ]:
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

for name, text in CORPUS.items():
    result = translate_traceback(text, translate)
    if result.failures:
        print(f"{name}: refused ({[f.reason for f in result.failures]}), nothing written")
        continue
    destination = OUTPUT_DIR / f"{name}.{TARGET_LANGUAGE}.txt"
    destination.write_text(result.text, encoding="utf-8")
    print("wrote", destination.name)

## What this recipe deliberately does not do

- It does not explain the error or suggest a fix. It translates one sentence.
  `examples/Regional_Code_Helper/` is the free-form assistant for that job; the
  two overlap in audience and in nothing else.
- It does not translate the `Traceback (most recent call last):` header or the
  two chain notes. Those are fixed interpreter boilerplate, identical in every
  traceback in the world; freezing them keeps the output recognisable as a Python
  traceback and keeps it pasteable into a search engine. Translating them is a
  static phrase table reviewed by a speaker of each language, not an API call.
- It does not fix the code, fetch answers from the internet, or speak the
  traceback aloud.
- It does not handle Java, JavaScript or C++ stack traces.

One measured cost, stated rather than hidden: in
`can only concatenate list (not "str") to list` the bare word `list` really is a
type name, and because `list` is excluded from the protected type words it will
be translated. That is the price of not freezing `list index out of range`
solid. `PROTECTED_TYPE_WORDS` in `traceback_translator.py` is a named constant
you can edit for your own corpus.